In [1]:
!pip install opencv-python

## Import knižníc
V tejto bunke importujeme všetky potrebné knižnice.

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os




## Vymazanie starých masiek z výstupného priečinka

Tento blok zabezpečí, že priečinok s maskami bude pred spracovaním prázdny.
Predchádza to miešaniu starých a nových výsledkov.

In [3]:
folder = 'data/Algorithmic_mask_creation/creat_mask_zasielka2'

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)

## Vytváranie masiek
- Na začiatku sa definujú cesty k vstupným orezaným obrázkom a k priečinku, kam sa budú ukladať masky.

Funkcia `connect_line`:
- Jednoducho spojí dva body priamkou.

Funkcia `connect_z_shape_dynamic`:
- Vytvorí dynamický „Z“ tvar — používa sa pri väčších medzerách, kde je vhodné vytvoriť ohyb, aby maska lepšie kopírovala tvar objektu.

Funkcia `repair_top_gaps`:
- rozdelí masku na niekoľko horizontálnych pásov,
- v každom pásme nájde najväčšie medzery,
- rozhodne, či ich spojiť priamkou alebo Z‑tvarom,
- zlúči výsledky späť do jednej masky.

Spracovanie každého obrázka:
Hlavná slučka prechádza všetky obrázky a vykonáva:
- Odstránenie vertikálnych čiar (napr. stĺpce, rušenie). Pomocou morfologických operácií sa detegujú a inpaintujú vertikálne rušivé línie.
- Detekcia červenej a žltej farby (v HSV). Vytvorí sa počiatočná maska na základe farieb, ktoré reprezentujú objekt.
- Odstránenie vertikálnych artefaktov z masky. Z masky sa odstránia dlhé vertikálne šmuhy, ktoré by mohli narušiť tvar objektu.
- Morfologické vyplnenie masky. Použije sa CLOSE a DILATE, aby sa maska vyhladila a vyplnili sa malé diery.
- Oprava medzier pomocou `repair_top_gaps`. Tu sa aplikujú všetky inteligentné opravy masky.
- Uloženie masky a vizualizácia Notebook zobrazí pôvodný obrázok, vytvorenú masku, prekrytie masky na obrázku.



In [ ]:
input_folder = 'data/CropPictures/orezane_zasielka2_new'
output_folder = 'data/Algorithmic_mask_creation/creat_mask_zasielka2'
os.makedirs(output_folder, exist_ok=True)

filenames = sorted([f for f in os.listdir(input_folder) if f.endswith('.jpg')])

def connect_line(mask, p1, p2):
    cv2.line(mask, p1, p2, 255, thickness=5)
    return mask

def connect_z_shape_dynamic(mask, p1, p2, offset_ratio=0.05, max_z_height=180, min_z_height=30, gap_length=None):
    x1, y1 = p1
    x2, y2 = p2
    dx = x2 - x1
    offset = int(abs(dx) * offset_ratio)
    height = int(gap_length / 2) if gap_length else int(abs(dx) / 2)
    height = min(max(height, min_z_height), max_z_height)
    mid1_x = x1 + offset if dx >= 0 else x1 - offset
    mid2_x = x2 - offset if dx >= 0 else x2 + offset
    mid_y = int((y1 + y2) / 2)
    z_points = [(x1, y1), (mid1_x, mid_y - height), (mid2_x, mid_y + height), (x2, y2)]
    for i in range(len(z_points) - 1):
        cv2.line(mask, z_points[i], z_points[i + 1], 255, thickness=5)
    return mask

def gaps_overlap(g1, g2, min_overlap_ratio=0.3):
    start1, end1 = g1['start'], g1['end']
    start2, end2 = g2['start'], g2['end']
    inter_start = max(start1, start2)
    inter_end = min(end1, end2)
    overlap = max(0, inter_end - inter_start)
    len1 = end1 - start1
    len2 = end2 - start2
    overlap_ratio1 = overlap / len1 if len1 > 0 else 0
    overlap_ratio2 = overlap / len2 if len2 > 0 else 0
    return overlap_ratio1 >= min_overlap_ratio and overlap_ratio2 >= min_overlap_ratio

def repair_top_gaps(mask, line_count=3, num_gaps=5, min_gap_for_connection=50, z_threshold=150):
    repaired_rows = []
    split_rows = np.array_split(mask, line_count, axis=0)
    gap_lines_by_row = []
    height_offsets = [0]
    for row in split_rows[:-1]:
        height_offsets.append(height_offsets[-1] + row.shape[0])
    for idx, (row, y_offset) in enumerate(zip(split_rows, height_offsets)):
        row_proj = np.sum(row, axis=0)
        binary_proj = (row_proj > 0).astype(np.int32)
        inverse_proj = 1 - binary_proj
        edges = np.where(np.diff(np.concatenate(([0], inverse_proj, [0]))) != 0)[0]
        gaps = [(edges[i], edges[i + 1]) for i in range(0, len(edges) - 1, 2)]
        gap_lengths = [end - start for start, end in gaps]
        row_gap_data = []
        if gap_lengths:
            top_indices = np.argsort(gap_lengths)[-num_gaps:][::-1]
            top_gaps = [gaps[i] for i in top_indices]
            for start, end in top_gaps:
                gap_length = end - start
                if gap_length < min_gap_for_connection:
                    continue
                left_slice = row[:, start - 5:start] if start >= 5 else row[:, :start]
                left_y = np.where(left_slice == 255)
                y1 = int(np.median(left_y[0])) if left_y[0].size > 0 else row.shape[0] // 2
                right_slice = row[:, end:end + 5] if end + 5 < row.shape[1] else row[:, end:]
                right_y = np.where(right_slice == 255)
                y2 = int(np.median(right_y[0])) if right_y[0].size > 0 else row.shape[0] // 2
                p1 = (start, y1 + y_offset)
                p2 = (end, y2 + y_offset)
                row_gap_data.append({
                    'start': start, 'end': end, 'length': gap_length,
                    'p1': p1, 'p2': p2, 'row_index': idx,
                    'z_candidate': gap_length >= z_threshold
                })
        gap_lines_by_row.append(row_gap_data)
        repaired_rows.append(row)
    full_mask = np.vstack(repaired_rows)
    grouped_gaps = []
    for row_gaps in gap_lines_by_row:
        for gap in row_gaps:
            matched = False
            for group in grouped_gaps:
                if gaps_overlap(gap, group[0]):
                    group.append(gap)
                    matched = True
                    break
            if not matched:
                grouped_gaps.append([gap])
    repaired_gaps = set()
    for group in grouped_gaps:
        by_row = [None] * line_count
        for g in group:
            by_row[g['row_index']] = g
        is_all_z = all(g and g['z_candidate'] for g in by_row)
        for i in range(line_count):
            g = by_row[i]
            if g:
                if is_all_z:
                    full_mask = connect_z_shape_dynamic(full_mask, g['p1'], g['p2'], gap_length=g['length'])
                else:
                    full_mask = connect_line(full_mask, g['p1'], g['p2'])
                repaired_gaps.add((g['row_index'], g['start'], g['end']))
    for row_idx, row_gaps in enumerate(gap_lines_by_row):
        for g in row_gaps:
            if (g['row_index'], g['start'], g['end']) not in repaired_gaps:
                full_mask = connect_line(full_mask, g['p1'], g['p2'])
    return full_mask

fig, axs = plt.subplots(len(filenames), 3, figsize=(24, 6 * len(filenames)))
if len(filenames) == 1:
    axs = np.expand_dims(axs, 0)

for i, fname in enumerate(filenames):
    image = cv2.imread(os.path.join(input_folder, fname))
    if image is None:
        print(f" Obrázok {fname} sa nepodarilo načítať.")
        continue

    original_image = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 30))
    vertical_lines = cv2.morphologyEx(gray, cv2.MORPH_OPEN, vertical_kernel, iterations=1)
    _, vertical_lines_bin = cv2.threshold(vertical_lines, 30, 255, cv2.THRESH_BINARY)
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    vertical_mask = cv2.dilate(vertical_lines_bin, kernel_dilate, iterations=1)
    image = cv2.inpaint(image, vertical_mask, 3, cv2.INPAINT_TELEA)

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lower_red1 = np.array([0, 100, 100])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([160, 100, 100])
    upper_red2 = np.array([179, 255, 255])
    lower_yellow = np.array([15, 100, 100])
    upper_yellow = np.array([30, 255, 255])
    mask_red1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask_red2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
    initial_mask = cv2.bitwise_or(mask_red1, mask_red2)
    initial_mask = cv2.bitwise_or(initial_mask, mask_yellow)

    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 50))
    vertical_lines = cv2.morphologyEx(initial_mask, cv2.MORPH_OPEN, vertical_kernel)
    vertical_lines = cv2.dilate(vertical_lines, np.ones((3, 3), np.uint8), iterations=1)
    initial_mask = cv2.subtract(initial_mask, vertical_lines)

    kernel_close = np.ones((10, 10), np.uint8)
    kernel_dilate = np.ones((5, 5), np.uint8)
    filled_mask = cv2.morphologyEx(initial_mask, cv2.MORPH_CLOSE, kernel_close)
    filled_mask = cv2.morphologyEx(filled_mask, cv2.MORPH_DILATE, kernel_dilate)

    repaired_mask_with_shape = repair_top_gaps(
        filled_mask.copy(),
        line_count=3,
        num_gaps=100,
        min_gap_for_connection=5,
        z_threshold=40
    )

    output_mask_path = os.path.join(output_folder, os.path.splitext(fname)[0] + '_mask.png')
    cv2.imwrite(output_mask_path, repaired_mask_with_shape)

    axs[i, 0].imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
    axs[i, 0].set_title('Original image', fontsize=25)
    axs[i, 0].axis('off')

    axs[i, 1].imshow(repaired_mask_with_shape, cmap='gray')
    axs[i, 1].set_title('Created mask', fontsize=25)
    axs[i, 1].axis('off')

    overlay_image = cv2.cvtColor(original_image.copy(), cv2.COLOR_BGR2RGB)
    mask_colored = np.zeros_like(overlay_image)
    mask_colored[:, :, 0] = repaired_mask_with_shape
    overlay = cv2.addWeighted(overlay_image, 0.7, mask_colored, 0.6, 0)

    axs[i, 2].imshow(overlay)
    axs[i, 2].set_title('The mask in the picture', fontsize=25)
    axs[i, 2].axis('off')

plt.tight_layout()
plt.show()
